In [1]:
import functools
from pathlib import Path

import altair
import polars
import util

file = util.notebook_file() if util.is_notebook() else __file__
tag = util.file_tag(file)
root_path = Path("..")
data_path = util.data_path(root_path)

In [2]:
full_tag = f"full_{tag}_calc"

In [3]:
sources = ["concentration", "experiment", "hill", "lokachari"]

# Read in data and rename species to match simulation
species_df = polars.read_csv(data_path / "cantera" / f"{full_tag}_species.csv")

data_dct: dict[str, polars.DataFrame] = {}
for source in sources:
    data = polars.read_csv(data_path / "hill" / f"{source}.csv")
    species_dct = {
        s0: s
        for s0, s in species_df[source, "name"].drop_nulls().iter_rows()
        if s0 in data
    }
    species_dct.update(
        {
            f"{s0}_err": f"{s}_err"
            for s0, s in species_dct.items()
            if f"{s0}_err" in data
        }
    )
    # data = data.with_columns()
    data_dct[source] = data.rename(species_dct, strict=True)

data = polars.read_csv(data_path / "cantera" / f"{full_tag}.csv")
data_dct["current"] = data

In [4]:
data_dct["current"]

phi,O2_molecules,CPT_frac,N2_frac,O2_frac,H2(2),O2(6),N2,HO2(8),CO(12),CO2(13),CH4(33),CH3CHO(41),C2H4(52),C3H6(131),C3H4O(165),C4H6(227),C4H8(253),C5H6(478),CPT(563),C6H6(970),OH(4),H2O(5),C5H8(522),C5H8O(825)rs
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3.0,0.12233,0.005,0.9825,0.0125,737.875995,8647.73946,978503.70709,1.93368,2246.103049,255.720432,273.024751,37.571506,1564.971428,124.550438,357.008762,21.568123,43.234763,57.455685,2301.524021,2.12289,0.001012,3192.949789,577.86768,17.536116
2.0,0.1835,0.005,0.97625,0.01875,953.937461,12082.306859,970308.07609,3.540389,4198.183176,604.594806,380.589015,50.325926,1881.644614,115.298087,333.137571,22.682507,34.699328,55.062156,1507.969199,2.849615,0.002097,5327.810915,578.382138,17.816176
1.75,0.20962,0.005,0.973571,0.021429,969.245034,14017.973408,967273.885821,4.074798,4639.867708,724.785206,382.524399,50.062512,1885.873175,105.462177,313.562988,22.005314,29.92024,55.311771,1362.169209,2.774499,0.002442,5896.874482,581.299456,18.016004
1.5,0.24456,0.005,0.97,0.025,950.569153,16924.051249,963498.520119,4.636088,4952.87736,849.238131,369.359189,47.605161,1848.845894,93.456093,289.545114,21.316401,24.826779,57.174102,1247.479955,2.606424,0.002774,6414.693073,594.130939,18.606333
1.25,0.29347,0.005,0.965,0.03,891.958187,21387.999991,958489.205653,5.200672,5085.948586,965.834376,341.171692,43.055942,1768.239041,80.151412,261.526442,20.830824,19.812789,61.388536,1163.980052,2.372454,0.003059,6845.072622,620.971046,19.752826
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.17,2.15785,0.005,0.774412,0.220588,210.047456,211572.876434,771171.530144,7.384873,3119.202041,708.206323,72.5262,10.488632,588.680801,15.338579,87.889076,18.067347,2.083454,136.881926,1039.50309,0.694179,0.00358,7405.0514,941.817779,33.246178
0.15,2.44557,0.005,0.745,0.25,189.583825,240973.730285,741981.161865,7.426817,3014.81684,665.500644,64.095325,9.680039,536.050427,13.922451,83.250527,17.841237,1.809653,140.124978,1038.198572,0.628414,0.003586,7383.301409,949.928993,33.598516
0.13,2.75147,0.005,0.706538,0.288462,168.430027,279407.570932,703776.940935,7.471368,2901.590738,618.791662,55.334961,8.835337,480.520232,12.483849,78.418262,17.596871,1.537685,143.444605,1036.846078,0.557262,0.003592,7356.319128,957.537888,33.94413


In [5]:
variable = "O2_molecules"
# species = "C5H8O(825)rs"
species = "C5H8(522)"

expt = (
    altair.Chart(data_dct["experiment"])
    .mark_circle()
    .encode(x="O2_molecules", y=species, color=altair.value("black"))
)

sim_df = functools.reduce(
    lambda x, y: x.join(y, on=variable),
    [
        data_dct[source][variable, species].rename({species: source})
        for source in ["lokachari", "hill", "current"]
    ],
)
print(sim_df)

expt + altair.Chart(sim_df).mark_line().transform_fold(
    fold=["lokachari", "hill", "current"],
).encode(x=variable, y="value:Q", color="key:N")

shape: (27, 4)
┌──────────────┬───────────┬───────┬────────────┐
│ O2_molecules ┆ lokachari ┆ hill  ┆ current    │
│ ---          ┆ ---       ┆ ---   ┆ ---        │
│ f64          ┆ f64       ┆ f64   ┆ f64        │
╞══════════════╪═══════════╪═══════╪════════════╡
│ 0.12233      ┆ 353.7     ┆ 478.7 ┆ 577.86768  │
│ 0.1835       ┆ 327.1     ┆ 461.8 ┆ 578.382138 │
│ 0.20962      ┆ 330.7     ┆ 458.6 ┆ 581.299456 │
│ 0.24456      ┆ 339.5     ┆ 461.7 ┆ 594.130939 │
│ 0.29347      ┆ 353.4     ┆ 474.3 ┆ 620.971046 │
│ …            ┆ …         ┆ …     ┆ …          │
│ 2.15785      ┆ 442.6     ┆ 666.6 ┆ 941.817779 │
│ 2.44557      ┆ 443.3     ┆ 672.3 ┆ 949.928993 │
│ 2.75147      ┆ 444.0     ┆ 677.9 ┆ 957.537888 │
│ 3.33486      ┆ 444.6     ┆ 683.1 ┆ 964.230246 │
│ 3.66835      ┆ 445.1     ┆ 685.5 ┆ 967.010499 │
└──────────────┴───────────┴───────┴────────────┘


alt.LayerChart(...)